In [5]:
import time
import os
import psutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from flaml import AutoML



In [6]:
df = pd.read_csv("../../data/creditcard.csv")

In [7]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [9]:
# =========================
# FUNÇÃO AUXILIAR DE RECURSOS
# =========================
def get_memory_usage_mb():
    """Retorna o uso atual de memória RAM do processo em Megabytes."""
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 2)

# =========================
# INSTALAR FLAML
# =========================
# %pip install "flaml[automl]" psutil

for SEED in [42, 101, 223, 302, 404]:
    print(f"\n================================")
    print(f"EXECUTANDO COM SEED: {SEED}")
    print(f"================================")

    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # =========================
    # TREINAMENTO COM FLAML + MÉTRICA COMPUTACIONAL
    # =========================
    automl = AutoML()
    settings = {
        "task": "classification",
        "metric": "f1", # Métrica orientada a custo/desbalanceamento
        "time_budget": 300,
        "log_file_name": "flaml_cost_sensitive.log",
        "seed": SEED
    }

    print("Iniciando treinamento com FLAML...")
    mem_antes_treino = get_memory_usage_mb()
    inicio_treino = time.perf_counter()

    automl.fit(X_train=X_train, y_train=y_train, **settings)

    fim_treino = time.perf_counter()
    tempo_treino_total = round(fim_treino - inicio_treino, 2)
    mem_depois_treino = get_memory_usage_mb()

    # =========================
    # RESULTADOS DO MELHOR MODELO
    # =========================
    print("\n===== MELHOR RESULTADO FLAML =====")
    print("Melhor modelo:", automl.best_estimator)
    print("Melhores parâmetros:", automl.best_config)
    print("Melhor loss:", automl.best_loss)

    # =========================
    # SELECIONAR OS 5 MELHORES ESTIMADORES
    # =========================
    perdas_por_estimador = automl.best_loss_per_estimator
    melhores_estimadores = sorted(
        perdas_por_estimador.items(),
        key=lambda item: item[1]
    )[:5]

    if not melhores_estimadores:
        raise RuntimeError("O FLAML não encontrou estimadores avaliados.")

    print("\n===== 5 MELHORES ESTIMADORES (VALIDAÇÃO FLAML) =====")
    print(pd.DataFrame(
        melhores_estimadores,
        columns=["Modelo", "Loss validação"]
    ).to_string(index=False))

    # =========================
    # MÉTRICAS DOS 5 MELHORES NO TESTE (COM BLINDAGEM E RECONSTRUÇÃO)
    # =========================
    resultados = []
    y_true = y_test.to_numpy()

    for rank, (nome_modelo, loss_validacao) in enumerate(melhores_estimadores, start=1):
        try:
            modelo = automl.best_model_for_estimator(nome_modelo)
            
            # Se o modelo retornado pelo FLAML for None ou não possuir motor interno treinado, recriamos via config otimizada
            if modelo is None or getattr(modelo, "_model", None) is None:
                params = automl.best_config_per_estimator.get(nome_modelo, {})
                
                if nome_modelo in ['xgboost', 'xgb_limitdepth']:
                    from xgboost import XGBClassifier
                    modelo = XGBClassifier(**params, random_state=SEED, n_jobs=-1)
                elif nome_modelo == 'lgbm':
                    from lightgbm import LGBMClassifier
                    modelo = LGBMClassifier(**params, random_state=SEED, n_jobs=-1, verbose=-1)
                elif nome_modelo == 'rf':
                    from sklearn.ensemble import RandomForestClassifier
                    modelo = RandomForestClassifier(**params, random_state=SEED, n_jobs=-1)
                elif nome_modelo == 'extra_tree':
                    from sklearn.ensemble import ExtraTreesClassifier
                    modelo = ExtraTreesClassifier(**params, random_state=SEED, n_jobs=-1)
                elif nome_modelo == 'sgd':
                    from sklearn.linear_model import SGDClassifier
                    modelo = SGDClassifier(**params, random_state=SEED, loss='log_loss')
                elif nome_modelo == 'lrl1':
                    from sklearn.linear_model import LogisticRegression
                    modelo = LogisticRegression(**params, random_state=SEED, penalty='l1', solver='liblinear')
                else:
                    print(f"Aviso: Estimador {nome_modelo} não mapeado. Pulando...")
                    continue
                
                # Treina o modelo reconstruído com os dados de treino
                modelo.fit(X_train, y_train)
            else:
                # Extrai o estimador interno scikit-learn/XGBoost se estiver envelopado pelo FLAML
                if hasattr(modelo, "estimator"):
                    modelo = modelo.estimator
                elif hasattr(modelo, "_model"):
                    modelo = modelo._model

            # Tempo de inferência para cada modelo
            inicio_inferencia = time.perf_counter()
            y_pred = modelo.predict(X_test)

            if hasattr(modelo, "predict_proba"):
                probabilidades = modelo.predict_proba(X_test)
                classes = list(modelo.classes_)
                y_score = probabilidades[:, classes.index(1)]
            elif hasattr(modelo, "decision_function"):
                y_score = modelo.decision_function(X_test)
                if list(modelo.classes_).index(1) == 0:
                    y_score = -y_score
            else:
                y_score = y_pred
                
            tempo_inferencia = time.perf_counter() - inicio_inferencia

            resultados.append({
                "Posição": rank,
                "Modelo": nome_modelo,
                "Loss validação": round(float(loss_validacao), 4),
                "Accuracy": round(accuracy_score(y_true, y_pred), 4),
                "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
                "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
                "F1-score Weighted": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
                "F1-score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
                "Average Precision": round(average_precision_score(y_true, y_score), 4),
                "Seed": SEED,
                "Tempo Inferência (s)": round(tempo_inferencia, 4),
                "Tempo Treino Total (s)": tempo_treino_total,
                "Uso RAM Treino (MB)": mem_depois_treino
            })
        except Exception as e:
            print(f"Erro ao avaliar o modelo {nome_modelo}: {e}")
            continue

    df_resultados_flaml = pd.DataFrame(resultados)
    print("\n===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====")
    print(df_resultados_flaml.to_string(index=False))


EXECUTANDO COM SEED: 42
Iniciando treinamento com FLAML...
[flaml.automl.logger: 11-08 20:29:26] {2470} INFO - task = classification
[flaml.automl.logger: 11-08 20:29:26] {2481} INFO - Evaluation method: holdout
[flaml.automl.logger: 11-08 20:29:26] {2584} INFO - Minimizing error metric: 1-f1
[flaml.automl.logger: 11-08 20:29:26] {2701} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 11-08 20:29:26] {3006} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 11-08 20:29:26] {3141} INFO - Estimated sufficient time budget=7694s. Estimated necessary time budget=178s.
[flaml.automl.logger: 11-08 20:29:26] {3192} INFO -  at 0.8s,	estimator lgbm's best error=1.0000e+00,	best estimator lgbm's best error=1.0000e+00
[flaml.automl.logger: 11-08 20:29:26] {3006} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 11-08 20:29:26] {3192} INFO -  at 0.8s,	estimator lgbm's best error=1.0000e

/opt/homebrew/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:34:26] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "FLAML_sample_size" } are not used.

  warnings.warn(smsg, UserWarning)


Erro ao avaliar o modelo extra_tree: ExtraTreesClassifier.__init__() got an unexpected keyword argument 'max_leaves'
Erro ao avaliar o modelo rf: RandomForestClassifier.__init__() got an unexpected keyword argument 'max_leaves'

===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====
 Posição         Modelo  Loss validação  Accuracy  Precision  Recall  F1-score Weighted  F1-score  Average Precision  Seed  Tempo Inferência (s)  Tempo Treino Total (s)  Uso RAM Treino (MB)
       1        xgboost          0.0541    0.9995     0.9277  0.7857             0.9995    0.8508             0.8625    42                0.0814                  301.39               453.64
       2 xgb_limitdepth          0.0541    0.9996     0.9294  0.8061             0.9995    0.8634             0.8553    42                0.0271                  301.39               453.64
       3           lgbm          0.0789    0.9996     0.9512  0.7959             0.9996    0.8667             0.8694    42                0.205

/opt/homebrew/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:39:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "FLAML_sample_size" } are not used.

  warnings.warn(smsg, UserWarning)


Erro ao avaliar o modelo extra_tree: ExtraTreesClassifier.__init__() got an unexpected keyword argument 'max_leaves'
Erro ao avaliar o modelo rf: RandomForestClassifier.__init__() got an unexpected keyword argument 'max_leaves'

===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====
 Posição         Modelo  Loss validação  Accuracy  Precision  Recall  F1-score Weighted  F1-score  Average Precision  Seed  Tempo Inferência (s)  Tempo Treino Total (s)  Uso RAM Treino (MB)
       1 xgb_limitdepth          0.0667    0.9997     0.9655  0.8571             0.9997    0.9081             0.8929   101                0.0557                  301.95               603.59
       2        xgboost          0.0789    0.9997     0.9655  0.8571             0.9997    0.9081             0.8913   101                0.0720                  301.95               603.59
       5           lgbm          0.1646    0.9994     0.8750  0.7857             0.9994    0.8280             0.8647   101                0.006

/opt/homebrew/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:44:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "FLAML_sample_size" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/homebrew/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:44:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "FLAML_sample_size" } are not used.

  warnings.warn(smsg, UserWarning)



===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====
 Posição         Modelo  Loss validação  Accuracy  Precision  Recall  F1-score Weighted  F1-score  Average Precision  Seed  Tempo Inferência (s)  Tempo Treino Total (s)  Uso RAM Treino (MB)
       1     extra_tree          0.1268    0.9995     0.9259  0.7653             0.9995    0.8380             0.7840   223                0.0335                  300.52               588.44
       3 xgb_limitdepth          0.1304    0.9994     0.9211  0.7143             0.9994    0.8046             0.7654   223                0.0083                  300.52               588.44
       4           lgbm          0.1549    0.9995     0.9268  0.7755             0.9995    0.8444             0.8518   223                0.7846                  300.52               588.44
       5        xgboost          0.1781    0.9996     0.9620  0.7755             0.9995    0.8588             0.8530   223                0.0377                  300.52             

/opt/homebrew/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:49:44] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "FLAML_sample_size" } are not used.

  warnings.warn(smsg, UserWarning)


Erro ao avaliar o modelo extra_tree: ExtraTreesClassifier.__init__() got an unexpected keyword argument 'max_leaves'
Erro ao avaliar o modelo rf: RandomForestClassifier.__init__() got an unexpected keyword argument 'max_leaves'

===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====
 Posição         Modelo  Loss validação  Accuracy  Precision  Recall  F1-score Weighted  F1-score  Average Precision  Seed  Tempo Inferência (s)  Tempo Treino Total (s)  Uso RAM Treino (MB)
       1        xgboost          0.1642    0.9996     0.9634  0.8061             0.9996    0.8778             0.8885   302                0.0502                  290.39               607.23
       2 xgb_limitdepth          0.1642    0.9996     0.9639  0.8163             0.9996    0.8840             0.8910   302                0.0327                  290.39               607.23
       3           lgbm          0.1818    0.9996     0.9412  0.8163             0.9996    0.8743             0.8903   302                0.057

/opt/homebrew/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:54:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "FLAML_sample_size" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/homebrew/lib/python3.11/site-packages/xgboost/core.py:158: UserWarning: [20:54:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "FLAML_sample_size" } are not used.

  warnings.warn(smsg, UserWarning)


Erro ao avaliar o modelo extra_tree: ExtraTreesClassifier.__init__() got an unexpected keyword argument 'max_leaves'
Erro ao avaliar o modelo rf: RandomForestClassifier.__init__() got an unexpected keyword argument 'max_leaves'

===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====
 Posição         Modelo  Loss validação  Accuracy  Precision  Recall  F1-score Weighted  F1-score  Average Precision  Seed  Tempo Inferência (s)  Tempo Treino Total (s)  Uso RAM Treino (MB)
       1           lgbm          0.0789    0.9995     0.9500  0.7755             0.9995    0.8539             0.8175   404                0.6041                  305.07               738.77
       2 xgb_limitdepth          0.0789    0.9995     0.9375  0.7653             0.9995    0.8427             0.8430   404                0.0152                  305.07               738.77
       3        xgboost          0.0811    0.9995     0.9351  0.7347             0.9994    0.8229             0.8357   404                0.028